# A Basic Discounted Cash Flow Valuation

Chapter 1 of {cite}`farevuu2018` showcases the structure and functionality of a
basic Discounted Cash Flow (DCF) Valuation for a real estate property.

Like in the vast majority of cases, the implementation of their DCF model is in
Microsoft Excel. However, in these notebooks, we will be implementing the same
methodology in Python, using the
[Rangekeeper](https://github.com/daniel-fink/rangekeeper) open source library.

In this notebook, the core computational objects (classes), their functionality,
and how they are composed together into a valuation model are outlined.

To do so, we will be replicating Table 1.1 from {cite}`farevuu2018`, which
describes this DCF:

```{figure} resources/FaREVuU-table1.1.jpg
---
width: 100%
name: FaREVuU-table1.1
---
Table 1.1 From {cite}`farevuu2018`
```

First, import the libraries used by this example. Use the current walkthrough
environment, or install Rangekeeper with its `calculations` extra and Jupyter.
Currency units and dates are explicit, so this example does not depend on the
computer's locale.

Import Libraries:

In [1]:
from datetime import date
import math

from rangekeeper.model.flux import Flow, Stream, MissingValueHandling
from rangekeeper.model.duration import Span, Frequency
from rangekeeper.model.measure import Quantity
from rangekeeper.model.distribution import Distribution
from rangekeeper.calculations import projection, financial
from rangekeeper.calculations.series import AlignmentJoin, ResamplingMethod

## The Foundational Elements of a Proforma

### A `Flow`
The core element of a Cash Flow analysis is the cash flow itself: a sequence of
movements of currency, either positive (*inflow*) or negative (*outflow*).
Each movement is associated with a date or period and an amount. A movement
could be a transaction, payment or transfer of currency. It does not identify
the parties involved.

A cash flow is also sometimes referred to as a *line item*, which designates
its subject, such as "Operational Expenses" or "Income from Building 2's Parking".

Rangekeeper represents this content with a `Flow` from `model.flux`. It holds
units and ordered Movement records. Flows can represent currency, energy,
mass or other quantities. A display name labels a line without changing its
underlying Flow.

####  Initializing a `Flow`

First, specify the currency used in our proforma. We use Australian dollars
(`AUD`). Each Flow carries its units; no machine locale setting is required.

In [2]:
currency = "AUD"
currency

'AUD'

Next we define a `Flow` object from a list of dates and amounts:

In [3]:
transactions = {
    date(2020, 1, 1): 100,
    date(2020, 1, 2): 200,
    date(2019, 1, 1): 300,
    date(2020, 12, 31): -100,
}

# Put the movements in chronological order before constructing the Flow.
dates = sorted(transactions)

cash_flow = Flow.from_events(
    dates=dates,
    magnitudes=[transactions[day] for day in dates],
    units=currency,
)

We can view the Flow by entering its variable name in a cell:

Tables show units beside each line-item name. `—` marks an absent movement,
`?` marks an unknown amount, and a known zero is displayed as `0.00`.

In [4]:
cash_flow

Period / date,Flow [AUD]
2019-01-01,300.00
2020-01-01,100.00
2020-01-02,200.00
2020-12-31,-100.00


We can also inspect the underlying Movement records and their dates. Expand the
first output to see the records, including their amounts and identities.

In [5]:
cash_flow.movements

(Movement.from_data({'id': 'a7e182da-06a0-4f16-9ba5-d60cfb38a85b', 'date': '2019-01-01', 'magnitude': 300}),
 Movement.from_data({'id': '5e212ecb-6007-4289-9f8d-db496b916708', 'date': '2020-01-01', 'magnitude': 100}),
 Movement.from_data({'id': 'bf9b5ac7-163e-4822-9f80-55113ab2449b', 'date': '2020-01-02', 'magnitude': 200}),
 Movement.from_data({'id': 'f01d251d-272e-47ed-a08a-821e63850a58', 'date': '2020-12-31', 'magnitude': -100}))

In [6]:
[movement.date for movement in cash_flow.movements]

[datetime.date(2019, 1, 1),
 datetime.date(2020, 1, 1),
 datetime.date(2020, 1, 2),
 datetime.date(2020, 12, 31)]

Its units can be inspected through the `units` property:

In [7]:
cash_flow.units

'AUD'

#### Overview of a `Flow`

A Flow has units and an ordered sequence of movements:

1. Movements can be positive, negative or zero. An unresolved amount is distinct
   from zero.
2. Each movement has a date or covers a period. Dated movements are ordered
   chronologically; period movements are ordered by coverage.
3. Operations return new content and leave the original Flow unchanged.

`display()` presents the Flow as a table. Its optional `name` supplies a label
for this view. Without a name, the label is simply `Flow`.

In [8]:
cash_flow.display(name="Operational Expenses")

Period / date,Operational Expenses [AUD]
2019-01-01,300.00
2020-01-01,100.00
2020-01-02,200.00
2020-12-31,-100.00


### A `Span`
Spans define intervals of time that encompass the movements of a Flow.
A Span includes its start and excludes its end. Its `periods()` method divides
that interval into the chosen frequency.

We hold the property for ten years and forecast one additional year. The
additional year's cash flow determines the sale proceeds. Keeping the holding
and forecast spans separate makes that distinction visible.

In [9]:
start_date = date(2001, 1, 1)
holding_years = 10

holding_span = Span.from_duration(
    name="Holding period",
    start=start_date,
    frequency=Frequency.YEAR,
    count=holding_years,
)

forecast_span = Span.from_duration(
    name="Forecast",
    start=start_date,
    frequency=Frequency.YEAR,
    count=holding_years + 1,
)

annual_periods = forecast_span.periods(frequency=Frequency.YEAR)
sale_period = holding_span.periods(frequency=Frequency.YEAR)[-1]

holding_span, forecast_span

(Span.from_data({'name': 'Holding period', 'start_inclusive': '2001-01-01', 'end_exclusive': '2011-01-01'}),
 Span.from_data({'name': 'Forecast', 'start_inclusive': '2001-01-01', 'end_exclusive': '2012-01-01'}))

Inspect the first and last forecast periods. The forecast includes 2001–2011;
the holding period ends before 2011.

In [10]:
annual_periods[0], annual_periods[-1]

(Period.from_data({'start_inclusive': '2001-01-01', 'end_exclusive': '2002-01-01'}),
 Period.from_data({'start_inclusive': '2011-01-01', 'end_exclusive': '2012-01-01'}))

### A `Projection`
A projection takes a quantity and casts it over a sequence of periods according
to a specified rule. There are two forms:

1. **Extrapolation** takes an initial quantity and extends it over a sequence
   of periods. We use `projection.extrapolate()`.
2. **Distribution** takes a total quantity and spreads it over a sequence of
   periods according to a distribution. We use `projection.distribute()`.

To match line 4, *Potential Gross Income*, in Table 1.1, we use compound
extrapolation. The first year receives the initial amount; growth starts
between the first and second years.

In [11]:
compounding_rate = 0.02

Let's now use the Flow and extrapolation operation to construct the
*Potential Gross Income* line item:

In [12]:
initial_income = Quantity(magnitude=100, units=currency)

potential_gross_income = projection.extrapolate(
    initial=initial_income,
    periods=annual_periods,
    method=projection.ProjectionMethod.COMPOUND,
    rate=compounding_rate,
)

potential_gross_income.display(name="Potential Gross Income")

Period / date,Potential Gross Income [AUD]
2001,100.00
2002,102.00
2003,104.04
2004,106.12
2005,108.24
2006,110.41
2007,112.62
2008,114.87
2009,117.17
2010,119.51


Similarly, we define the 'Vacancy' line item by multiplying the movements of the
'Potential Gross Income' `Flow` by a vacancy rate:

In [13]:
vacancy_rate = 0.05

vacancy = potential_gross_income.scale(
    factor=vacancy_rate,
).negate()

vacancy.display(name="Vacancy Allowance")

Period / date,Vacancy Allowance [AUD]
2001,-5.00
2002,-5.10
2003,-5.20
2004,-5.31
2005,-5.41
2006,-5.52
2007,-5.63
2008,-5.74
2009,-5.86
2010,-5.98


```{note}
Note the sign of the movements of the 'Vacancy Allowance' `Flow` -- it is
negative, because it is an *outflow*.
```

### A `Stream`
A Stream groups labelled Flows into a table. It can resample their movements
to chosen periods and combine the selected lines.

Let's use the *Effective Gross Income* line item in Table 1.1 to illustrate
the concept. Both constituent Flows already use the same annual periods.

In [14]:
effective_gross_income = Stream(
    flows={
        "Potential Gross Income": potential_gross_income,
        "Vacancy Allowance": vacancy,
    },
)

effective_gross_income.display()

Period / date,Potential Gross Income [AUD],Vacancy Allowance [AUD]
2001,100.00,-5.00
2002,102.00,-5.10
2003,104.04,-5.20
2004,106.12,-5.31
2005,108.24,-5.41
2006,110.41,-5.52
2007,112.62,-5.63
2008,114.87,-5.74
2009,117.17,-5.86
2010,119.51,-5.98


The table keeps each line's label and units. To aggregate the constituent
Flows, we sum their amounts within each period into a resulting Flow:

In [15]:
effective_gross_income_flow = effective_gross_income.sum()

effective_gross_income_flow.display(name="Effective Gross Income")

Period / date,Effective Gross Income [AUD]
2001,95.00
2002,96.90
2003,98.84
2004,100.81
2005,102.83
2006,104.89
2007,106.99
2008,109.13
2009,111.31
2010,113.53


The result retains the annual periods. Period coverage does not automatically
record a payment date; we specify the valuation timing later.

`sum()` returns the combined Flow. When coverage information is also needed,
`aggregate()` returns the Flow together with that information.

#### Resampling movements
`resample()` groups each line's movements into new periods. `sum()` combines
lines within each period. Reuse the original dated transactions to see annual
resampling in action:

In [16]:
transaction_span = Span.from_duration(
    name="Transaction example",
    start=date(2019, 1, 1),
    frequency=Frequency.YEAR,
    count=2,
)

annual_transactions = Stream(
    flows={"Operational Expenses": cash_flow},
).resample(
    periods=transaction_span.periods(frequency=Frequency.YEAR),
    method=ResamplingMethod.SUM,
)

annual_transactions.display()

Period / date,Operational Expenses [AUD]
2019,300.00
2020,200.00


The annual amounts are 300 for 2019 and 200 for 2020. The original dated Flow
remains unchanged. We now return to the property forecast.

With this in mind, we can complete Table 1.1

In [17]:
opex_pgi_ratio = 0.35

operating_expenses = potential_gross_income.scale(
    factor=opex_pgi_ratio,
).negate()

net_operating_income = Stream(
    flows={
        "Effective Gross Income": effective_gross_income_flow,
        "Operating Expenses": operating_expenses,
    },
)

net_operating_income.display()

Period / date,Effective Gross Income [AUD],Operating Expenses [AUD]
2001,95.00,-35.00
2002,96.90,-35.70
2003,98.84,-36.41
2004,100.81,-37.14
2005,102.83,-37.89
2006,104.89,-38.64
2007,106.99,-39.42
2008,109.13,-40.20
2009,111.31,-41.01
2010,113.53,-41.83


In [18]:
capex_pgi_ratio = 0.10

capital_expenditures = potential_gross_income.scale(
    factor=capex_pgi_ratio,
).negate()

net_annual_cashflows = Stream(
    flows={
        "Net Operating Income": net_operating_income.sum(),
        "Capital Expenditures": capital_expenditures,
    },
)

net_annual_cashflow = net_annual_cashflows.sum()

net_annual_cashflows.display()

Period / date,Net Operating Income [AUD],Capital Expenditures [AUD]
2001,60.00,-10.00
2002,61.20,-10.20
2003,62.42,-10.40
2004,63.67,-10.61
2005,64.95,-10.82
2006,66.24,-11.04
2007,67.57,-11.26
2008,68.92,-11.49
2009,70.30,-11.72
2010,71.71,-11.95


The first year's net cash flow is `100 - 5 - 35 - 10 = 50`.

### Reversion
To calculate the sale proceeds, we capitalize the following year's net cash
flow **after capital expenditure**, as in Table 1.1. The last forecast movement
supplies that year 11 amount. We receive the proceeds in `sale_period`, the
final year of ownership.

A distribution projection places the capitalized total into that one period.
The uniform distribution preserves the total; with only one period, the whole
amount goes into the sale period.

In [19]:
exit_caprate = 0.05

next_year_cashflow = net_annual_cashflow.movements[-1].number
reversion_value = next_year_cashflow / exit_caprate

reversion_flow = projection.distribute(
    quantity=Quantity(
        magnitude=reversion_value,
        units=currency,
    ),
    periods=(sale_period,),
    distribution=Distribution.uniform(),
)

reversion_flow.display(name="Reversion")

Period / date,Reversion [AUD]
2010,"1,218.99"


Finally, we combine the net and reversion cash flows to calculate the project's
complete cash flows. We place both lines onto the annual forecast periods.

We explicitly choose zero for empty periods: no sale proceeds occur outside
the final holding year. This is a statement about this example's cash flows;
an unknown amount is not the same as an absent movement.

In [20]:
net_cashflows_with_reversion = Stream(
    flows={
        "Net Annual Cashflows": net_annual_cashflow,
        "Reversion": reversion_flow,
    },
).resample(
    periods=annual_periods,
    method=ResamplingMethod.SUM,
    missing=MissingValueHandling.ZERO,
)

net_cashflows_with_reversion.display()

Period / date,Net Annual Cashflows [AUD],Reversion [AUD]
2001,50.00,0.00
2002,51.00,0.00
2003,52.02,0.00
2004,53.06,0.00
2005,54.12,0.00
2006,55.20,0.00
2007,56.31,0.00
2008,57.43,0.00
2009,58.58,0.00
2010,59.75,"1,218.99"


Keep only the holding period's cash flows before valuation. The extra forecast
year supports the sale calculation but is outside our ownership period.

In [21]:
net_cashflows_with_reversion = net_cashflows_with_reversion.trim(
    span=holding_span,
)

project_cashflows = net_cashflows_with_reversion.sum()

net_cashflows_with_reversion.display()

Period / date,Net Annual Cashflows [AUD],Reversion [AUD]
2001,50.00,0.00
2002,51.00,0.00
2003,52.02,0.00
2004,53.06,0.00
2005,54.12,0.00
2006,55.20,0.00
2007,56.31,0.00
2008,57.43,0.00
2009,58.58,0.00
2010,59.75,"1,218.99"


### The complete proforma
To reproduce Table 1.1, construct a Stream containing the named line items.
`UNION` includes every period represented by any line, so we can show the
eleven-year forecast alongside the ten-year holding period. `transpose=True`
puts line items down the page and years across it.

Deductions are negative movements here. An absent movement appears as `—`,
while an unknown amount appears as `?`. The report contains both components
and subtotals; summing the entire table would double-count amounts.

In [22]:
table = Stream(
    flows={
        "Potential Gross Income": potential_gross_income,
        "Vacancy Allowance": vacancy,
        "Effective Gross Income": effective_gross_income_flow,
        "Operating Expenses": operating_expenses,
        "Net Operating Income": net_operating_income.sum(),
        "Capital Expenditures": capital_expenditures,
        "Net Annual Cashflows": net_annual_cashflow,
        "Reversion": reversion_flow,
        "Net Cashflow with Reversion": project_cashflows,
    },
    join=AlignmentJoin.UNION,
)

table.display(transpose=True)

Line item,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011
Potential Gross Income [AUD],100.00,102.00,104.04,106.12,108.24,110.41,112.62,114.87,117.17,119.51,121.90
Vacancy Allowance [AUD],-5.00,-5.10,-5.20,-5.31,-5.41,-5.52,-5.63,-5.74,-5.86,-5.98,-6.09
Effective Gross Income [AUD],95.00,96.90,98.84,100.81,102.83,104.89,106.99,109.13,111.31,113.53,115.80
Operating Expenses [AUD],-35.00,-35.70,-36.41,-37.14,-37.89,-38.64,-39.42,-40.20,-41.01,-41.83,-42.66
Net Operating Income [AUD],60.00,61.20,62.42,63.67,64.95,66.24,67.57,68.92,70.30,71.71,73.14
Capital Expenditures [AUD],-10.00,-10.20,-10.40,-10.61,-10.82,-11.04,-11.26,-11.49,-11.72,-11.95,-12.19
Net Annual Cashflows [AUD],50.00,51.00,52.02,53.06,54.12,55.20,56.31,57.43,58.58,59.75,60.95
Reversion [AUD],—,—,—,—,—,—,—,—,—,"1,218.99",—
Net Cashflow with Reversion [AUD],50.00,51.00,52.02,53.06,54.12,55.20,56.31,57.43,58.58,"1,278.75",—


### Present value
Given the annual discount rate of 7%, we can calculate the property's present
value. The first receipt occurs one annual period after valuation. The final
receipt, including sale proceeds, receives ten periods of discounting.

`calculate_pv()` discounts each movement and retains its period coordinates.
We then `collapse()` those discounted amounts into one movement on the
valuation date.

In [23]:
discount_rate = 0.07

pvs = financial.calculate_pv(
    flow=project_cashflows,
    rate=discount_rate,
    first_period=1,
)

pvs.display(name="Present Value")

Period / date,Present Value [AUD]
2001,46.73
2002,44.55
2003,42.46
2004,40.48
2005,38.59
2006,36.78
2007,35.07
2008,33.43
2009,31.87
2010,650.05


In [24]:
valuation_date = holding_span.start_inclusive

property_value = pvs.collapse(on=valuation_date)

property_value.display(name="Property Present Value")

Period / date,Property Present Value [AUD]
2001-01-01,"1,000.00"


In [25]:
property_pv = property_value.movements[0].number

print(f"Property PV: ${property_pv:,.0f}")

Property PV: $1,000


### Check the example
Check three points that follow from the example: ten years of receipts, sale
proceeds in the final holding period, and the $1,000 value shown in Table 1.1.
The value check applies to these original assumptions; changing them can change
the expected result.

In [26]:
assert len(project_cashflows.movements) == holding_years
assert reversion_flow.movements[0].period == sale_period
assert math.isclose(property_pv, 1000.0, abs_tol=0.01)

### Next steps
We have constructed a DCF from Flows, coordinated them through Streams, and
used Spans to define the forecast and holding periods. We calculated the result
from known assumptions.

The same financial relationships can later be declared as equations in a Model.
A Specification selects which variables are fixed and which are solved. These
Stream calculations do not themselves create those equations.